# Misc Code For Visualization

In [ ]:
import base64
from IPython.display import Image, display

def mm_ink(graph: str):
    graphbytes = graph.encode("utf-8")
    base64_string = base64.urlsafe_b64encode(graphbytes).decode("ascii")
    return "https://mermaid.ink/img/" + base64_string

def mm(graph: str):
    display(Image(url=mm_ink(graph)))

# Build a Rambam Q&A Chatbot with LlamaIndex + OpenAI + Gradio

In this notebook, we build a source-grounded Q&A chatbot over Rambam’s *Mishneh Torah*.


# Big Picture: What Is RAG?


# Tools we will use


- **LlamaIndex** for indexing and retrieval
    https://developers.llamaindex.ai/python/framework/
  - **OpenAI API** for embeddings and answer generation  
- **Gradio** for the chat interface  

---

In [ ]:
# @title
from IPython.display import Markdown,display
display(Markdown("# RAG Flow for Rambam Q&A"))

mm("""
flowchart TD
    A["User Question<br/>What does Rambam say about repentance?"]
    A --> B["Embed Query<br/>q = [0.82, -0.14, 0.33, 0.71]"]
    B --> C["Vector Search<br/>Find K Most Similar Chunks"]

    D1["Rambam Chunk 1<br/>Hilchot Teshuva 2:1"]
    D2["Rambam Chunk 2<br/>Hilchot Deot 1:3"]
    D3["Rambam Chunk 3<br/>Hilchot Teshuva 7:4"]

    D1 --> C
    D2 --> C
    D3 --> C

    C --> E["Return Top K Matches"]
    E --> F["Prompt to LLM<br/>Question + Sources"]
    F --> G["Final Answer<br/>with citations"]
""")

# RAG Flow for Rambam Q&A

# Load DataFrame

In [5]:
import pandas as pd

In [48]:
df = pd.read_parquet('rambam_df.parquet')

In [52]:
df.groupby(['section','chapter']).head(1)

,ref,section,section_type,chapter,number,text
0,"Mishneh Torah, Transmission of the Oral Law 1",Transmission of the Oral Law,intro,0,1,"<b>The Rambam's Introduction</b><sup class=""fo..."
45,"Mishneh Torah, Positive Mitzvot 1",Positive Mitzvot,positive_mitzvah,0,0,The first of the positive commandments is the ...
293,"Mishneh Torah, Negative Mitzvot 1",Negative Mitzvot,negative_mitzvah,0,0,The first mitzvah of the negative commandments...
663,"Mishneh Torah, Overview of Mishneh Torah Conte...",Overview of Mishneh Torah Contents,overview,0,1,I have seen fit to divide this work into fourt...
677,"Mishneh Torah, Foundations of the Torah 1:1",Foundations of the Torah,halacha,1,1,The foundation of all foundations and the pill...
...,...,...,...,...,...,...
15690,"Mishneh Torah, Kings and Wars 8:1",Kings and Wars,halacha,8,1,When the army's troops enter the territory of ...
15701,"Mishneh Torah, Kings and Wars 9:1",Kings and Wars,halacha,9,1,Six precepts were commanded to Adam:<br>a) the...
15715,"Mishneh Torah, Kings and Wars 10:1",Kings and Wars,halacha,10,1,A Noachide who inadvertently violates one of h...
15727,"Mishneh Torah, Kings and Wars 11:1",Kings and Wars,halacha,11,1,"In the future, the Messianic king will arise a..."


In [56]:
from IPython.display import display, HTML
display(HTML(df.text[0]))


# Installs Needed
* Maybe need more if not using Colab


In [7]:
!pip install llama_index openai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, whi

#  What is LlamaIndex

* https://developers.llamaindex.ai/python/framework/


# Load OpenAI API Keys + Set as os environment variable

In [1]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Configure OpenAI + LlamaIndex

We use OpenAI in two places:

1. **Embeddings** — turning Rambam text and user questions into vectors  
2. **LLM responses** — generating the final answer from retrieved sources

The defaults are pretty good, but I find that we are much better off using the larger embedding model!!!

In [2]:
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings



Settings.llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")

## Convert DataFrame Rows into LlamaIndex Documents

Each row in the dataframe becomes a `Document`.

The `text` field becomes the content.

The other fields become metadata, which lets us cite sources later.

**CODE MYSELF**

In [29]:
from llama_index.core import Document
def df_to_documents(df):
    documents = []

    for row in df.itertuples(index=False):
        metadata = {
            "ref": row.ref,
            "section": row.section,
            "section_type": row.section_type,
            "chapter": row.chapter,
            "number": row.number,

        }
        documents.append(
            Document(
                text=row.text,
                metadata=metadata
            )
        )

    return documents

documents = df_to_documents(df)

len(documents), documents[0]

(15741,
 Document(id_='caa8f732-770b-472f-bc15-06ee1cee5390', embedding=None, metadata={'ref': 'Mishneh Torah, Transmission of the Oral Law 1', 'section': 'Transmission of the Oral Law', 'section_type': 'intro', 'chapter': 0, 'number': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='<b>The Rambam\'s Introduction</b><sup class="footnote-marker">1</sup><i class="footnote">The heading "Introduction" is not found in any of the manuscript editions of the <i>Mishneh Torah</i> and appears to be a printer\'s addition. Note <i>Hilchot Shechitah</i> 1:4, where the Rambam refers to "...the Oral Law, which is called `the mitzvah,\' as we explained in the beginning of this text."<br>By referring to these passages as "the beginning" of the text and not "the introduction to the text," the Rambam implies that the subject matter contained in th

## Build the Vector Index

In [32]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_documents(documents,show_progress=True)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2026 [00:00<?, ?it/s]

# Optional, but recommended to Save Embeddings/Indexing for Later Querying. Should Save to VectorStore Like ChromaDB. Llama Index has tools for all this

## Save and Load the Index

Persisting the index prevents us from rebuilding embeddings every time.


```
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)

index.storage_context.persist(persist_dir="./storage")
```
Load later
```
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(
    persist_dir="./storage"
)

index = load_index_from_storage(storage_context)
```

### Let's see a sample embedding

In [42]:
vs = index.storage_context.vector_store.data.embedding_dict
len(next(iter(vs.values())))

3072

## 🔹 Query Engine: “Ask → Answer” All in One



In [44]:
query_engine = index.as_query_engine()
response = query_engine.query("What does Rambam say about free will?")

In [46]:
response.response

"Rambam asserts that free will is granted to all individuals, allowing them the choice to pursue either a righteous or wicked path. He emphasizes that each person has the capacity to know good and evil and to act according to their desires without external compulsion. This ability to choose is fundamental to human nature and is essential for moral responsibility. Rambam argues that if individuals were predetermined to be righteous or wicked, it would undermine the purpose of the Torah and the justice of divine retribution. Ultimately, he maintains that while everything occurs according to God's will, humans are still accountable for their actions, as they possess the freedom to make choices."

In [58]:
len(response.source_nodes)

2

**Retriever (Finding most relevant nodes) + Prompt + LLM → Final Answer**

---

### ⚙️ Customizing the Query Engine



In [59]:
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query("What does Rambam say about free will?")


In [60]:
len(response.source_nodes)

5



You control:

* `text_qa_template` → how the answer is written (grounded, cited, etc.)
* `response_mode` → short vs structured answers
* `similarity_top_k` → how many sources are passed in
* `node_postprocessors` → reranking / refinement

👉 This controls **how the answer is generated**

---

## 🔍 Retriever — “Find the Right Sources” only

In [61]:
retriever = index.as_retriever(similarity_top_k=5)
nodes = retriever.retrieve("What does Rambam say about free will?")

In [62]:
nodes

[NodeWithScore(node=TextNode(id_='39768992-916f-4994-92e6-d750a945976d', embedding=None, metadata={'ref': 'Mishneh Torah, Repentance 5:1', 'section': 'Repentance', 'section_type': 'halacha', 'chapter': 5, 'number': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a72bf164-316e-4630-bbc1-11e11bd61b37', node_type='4', metadata={'ref': 'Mishneh Torah, Repentance 5:1', 'section': 'Repentance', 'section_type': 'halacha', 'chapter': 5, 'number': 1}, hash='82acaae479a438b419f0a26e7d80b12c3aaf07bb63a06f384b0e3eff2afd19f3')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Free will is granted to all men. If one desires to turn himself to the path of good and be righteous, the choice is his. Should he desire to turn to the path of evil and be wicked, the choice is his.<br>This is [the intent of] the Torah\'s statement (Genesis 3:22 : "Behold, man has become unique as ourselves, knowing 

---

### ⚙️ Customizing the Retriever further

```python
from llama_index.core import MetadataFilters, ExactMatchFilter

retriever = index.as_retriever(
    similarity_top_k=5,
    filters=MetadataFilters(filters=[
        ExactMatchFilter(key="hilchot", value="Repentance")
    ])
)
```

You control:

* how many results (`top_k`)
* which parts of Rambam are searched
* filtering by metadata (e.g. Sefer HaMadda only)

👉 This controls **what information enters the system**

---

## 🎯 Key Insight

> The retriever does **not generate the answer**
> but it **determines what goes into the prompt**

So:

> **Better retrieval → better sources → better answer**


# Custom Grounded Answer Function (Instead of using using .query)

Now we manually build the RAG pipeline:

1. Retrieve relevant Rambam sources
2. Insert sources into a prompt
3. Ask the LLM to answer using only those sources
4. Return both the answer and source table


In [63]:
# Step 8 — Grounded answer function
from openai import OpenAI
from llama_index.core.schema import MetadataMode

client = OpenAI()


def retrieve_sources(index, query, k=5):
    retriever = index.as_retriever(similarity_top_k=k)
    nodes = retriever.retrieve(query)

    rows = []

    for node in nodes:
        md = node.metadata or {}

        rows.append({
            "ref": md.get("ref"),
            "hilchot": md.get("hilchot"),
            "chapter": md.get("chapter"),
            "halacha": md.get("halacha"),
            "score": getattr(node, "score", None),
            "text": node.get_content(metadata_mode=MetadataMode.NONE),
        })

    return pd.DataFrame(rows), nodes


def ask_rambam_with_sources(index, question, k=5):
    results_df, nodes = retrieve_sources(index, question, k=k)

    if results_df.empty:
        return {
            "answer": "I could not retrieve any relevant Rambam source.",
            "sources": results_df,
        }

    sources = "\n\n".join(
        f"{row['ref']}\n{row['text']}"
        for _, row in results_df.iterrows()
    )

    prompt = f'''
You are a Rambam assistant.

Answer in clear English.
Use ONLY the sources below.
Cite the references clearly.
Do not invent sources.
If the sources only partially answer the question, say so.

Question:
{question}

Sources:
{sources}
'''

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": results_df,
    }


In [64]:
test = ask_rambam_with_sources(
    index,
    "What does Rambam say about free will?",
    k=5
)

print(test["answer"])
test["sources"]


Rambam, in his Mishneh Torah, discusses the concept of free will extensively, emphasizing that it is a fundamental principle of the Torah and mitzvot. He asserts that free will is granted to all individuals, allowing them to choose their path, whether it be good or evil (Mishneh Torah, Repentance 5:1). This ability to choose is what makes humans unique, as they can act according to their own initiative and desires without being compelled by any external force.

Rambam argues against the notion that God decrees whether a person will be righteous or wicked at the time of their creation. Instead, each person has the potential to be as righteous as Moses or as wicked as Jeroboam, and it is their own choice and initiative that determine their path (Mishneh Torah, Repentance 5:2). He emphasizes that no one compels or leads a person towards a particular path; rather, it is the individual's decision.

Furthermore, Rambam explains that if God had predetermined a person's behavior, the commandme

,ref,hilchot,chapter,halacha,score,text
0,"Mishneh Torah, Repentance 5:1",None,5,None,0.663053,Free will is granted to all men. If one desire...
1,"Mishneh Torah, Repentance 5:4",None,5,None,0.619322,Were God to decree that an individual would be...
2,"Mishneh Torah, Repentance 5:3",None,5,None,0.611243,This principle is a fundamental concept and a ...
3,"Mishneh Torah, Human Dispositions 1:2",None,1,None,0.609083,"Nevertheless, these are merely tendencies, and..."
4,"Mishneh Torah, Repentance 5:2",None,5,None,0.606872,A person should not entertain the thesis held ...


# Gradio
https://www.gradio.app


## Chat Interface Example

Your function must accept:

* `message` → the latest user input
* `history` → previous messages (conversation state)

```python
def chat_fn(message, history):
    ...
```

## Gradio Chat UI

Now we wrap the Q&A function in a simple chat interface.


In [ ]:
import gradio as gr
def chat_fn(message,history):
    return ask_rambam_with_sources(
        index,
        message,
        k=5
    )["answer"]

chat_interface = gr.ChatInterface(
    fn=chat_fn)
chat_interface.launch(debug=True,share=True)


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3c23045f43ac004d56.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Prettier, More Custom Gradio App Version

In [ ]:
import os
import pandas as pd
import gradio as gr

# =========================
# Theme / CSS
# =========================
theme = gr.themes.Soft().set(
    body_background_fill="#f8f5ef",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#e5dccb",
    button_primary_background_fill="#7a5c2e",
    button_primary_background_fill_hover="#644a24",
    button_primary_text_color="#ffffff",
    input_background_fill="#fffdf8",
)

css = """
.gradio-container {
    max-width: 1100px !important;
    margin: 0 auto !important;
}

#app-title {
    text-align: center;
    margin-bottom: 0.25rem;
}

#app-subtitle {
    text-align: center;
    color: #6b6257;
    margin-bottom: 1.25rem;
}

.rambam-card {
    border: 1px solid #e5dccb;
    border-radius: 18px;
    padding: 18px;
    background: linear-gradient(180deg, #fffdf8 0%, #faf6ee 100%);
    box-shadow: 0 6px 18px rgba(60, 40, 10, 0.06);
}

footer {display: none !important;}

.gr-button-primary {
    border-radius: 12px !important;
    font-weight: 600 !important;
}

textarea, .gr-text-input, .gr-textbox {
    border-radius: 12px !important;
}

.gr-chatbot {
    border-radius: 16px !important;
    border: 1px solid #e5dccb !important;
}

.source-box {
    font-size: 0.95rem;
    line-height: 1.5;
    color: #3d372f;
}
"""

# =========================
# Helpers
# =========================
def build_sources_markdown(results_df):
    if results_df is None or results_df.empty:
        return ""

    parts = []
    for row in results_df.itertuples(index=False):
        ref = getattr(row, "ref", "Unknown ref")
        text = getattr(row, "text", "")
        parts.append(f"**{ref}**\n{text}")

    return "\n\n---\n### Sources\n" + "\n\n".join(parts)


def retrieve_sources_safe(question, k=5):
    results_df, nodes = retrieve_sources(index,question, k=k)

    if results_df is None:
        results_df = pd.DataFrame(columns=["ref", "text"])

    if "ref" not in results_df.columns:
        results_df["ref"] = ""
    if "text" not in results_df.columns:
        results_df["text"] = ""

    return results_df, nodes


# =========================
# Streaming retrieval + LLM
# =========================
def ask_rambam_with_sources_stream(question, k=5):
    question = str(question).strip()

    results_df, _ = retrieve_sources_safe(question, k=k)

    if results_df.empty:
        context_text = "No Rambam sources were retrieved."
    else:
        context_text = "\n\n".join(
            f"REF: {row.ref}\nTEXT: {row.text}"
            for row in results_df.itertuples(index=False)
        )

    system_prompt = """
You are a helpful assistant answering questions about the Rambam.

Rules:
1. Answer ONLY from the provided Rambam sources.
2. If the sources do not clearly answer the question, say so clearly.
3. Be concise, clear, and faithful to the source text.
4. Do not invent facts, citations, or quotations.
5. Write in English unless the user asks otherwise.
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved Rambam sources:
{context_text}

Please answer based only on these sources.
""".strip()

    accumulated = ""

    with client.responses.stream(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    ) as stream:
        for event in stream:
            if event.type == "response.output_text.delta":
                delta = event.delta
                accumulated += delta
                yield accumulated

        final_response = stream.get_final_response()

    final_text = getattr(final_response, "output_text", None) or accumulated
    final_text += build_sources_markdown(results_df)
    yield final_text


# =========================
# ChatInterface handler
# IMPORTANT: this must YIELD for streaming
# =========================
def respond(message, history):
    if not str(message).strip():
        yield "Please enter a question."
        return

    try:
        for partial in ask_rambam_with_sources_stream(message, k=5):
            yield partial
    except Exception as e:
        yield f"Error: {e}"


# =========================
# UI
# =========================
with gr.Blocks(theme=theme, css=css) as demo:
    gr.Markdown("# Ask the Rambam", elem_id="app-title")
    gr.Markdown(
        "A source-grounded Rambam search app using Sefaria + semantic search.",
        elem_id="app-subtitle",
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=600,
                show_label=False,
                avatar_images=(None, None),
            )

            gr.ChatInterface(
                fn=respond,          # streaming stays because respond() yields
                chatbot=chatbot,     # custom styled chatbot
            )

        with gr.Column(scale=1):
            gr.Markdown(
                """
<div class="rambam-card source-box">

### Tips
- Ask in plain English
- Try:
  - free will
  - repentance
  - Chol HaMoed work
  - returning lost objects

### Notes
Answers are grounded in Rambam sources.

</div>
                """
            )

demo.launch(
    prevent_thread_lock=True,
    debug=True,
    share=True
)s

/tmp/ipykernel_27251/4038549281.py:175: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=css) as demo:
/tmp/ipykernel_27251/4038549281.py:175: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=css) as demo:
/tmp/ipykernel_27251/4038549281.py:184: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_27251/4038549281.py:184: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e199f1108203fa0ce3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
